# Big project: Patient-nurse-room assignment

In this project, you are required to solve a healthcare scheduling problem involving the assignment of patients to rooms and nurses to rooms in a hospital. The goal is to develop an optimization model using mip that minimizes the total delay in patient admissions while adhering to a set of operational and resource constraints. This problem is inspired by the Integrated Healthcare Timetabling Competition 2024 (https://ihtc2024.github.io/).

The hospital operates with a set of patients, rooms, and nurses, each with specific characteristics. Patients are defined by their release date, indicating the earliest day they can be admitted, and a due date, specifying the latest allowable admission day. Patients also have a required length of stay. Rooms are characterized by fixed capacities that determine the maximum number of patients they can accommodate at any given time. Additionally, some patients may have incompatibilities with certain rooms, restricting their assignment.

Nurses have predefined rosters detailing their availability across days. Each occupied room must have one assigned nurse on every day of the scheduling horizon. Nurses can be assigned to a maximum of 3 rooms in a given day.

The problem requires assigning each patient to a room and ensuring that every occupied room has an assigned nurse during each day. These assignments must satisfy several constraints: no room may exceed its capacity, nurse assignments must respect their availability, and patients must only be assigned to compatible rooms. The primary objective is to minimize the total delay in admissions, defined as the sum of the differences between actual admission dates and the release dates of all patients. In what follows we detail the constraints and decisions.


**Constraints:**

- A patient’s length of stay must be respected, meaning they occupy a room for the full duration of their stay.
- A room cannot exceed its capacity on any given day.
- Patients can only be assigned to rooms that they are compatible with.
- Each occupied room must have exactly one assigned nurse per day.
- Nurses can only be assigned to rooms on days when they are available.
- Nurses can be assigned to a maximum of 3 different rooms in the same day.
- Admission days must be within the specified release and due dates.

**Decisions:**

- Assign each patient to a room, ensuring that the same room is assigned for the duration of their stay.
- Determine the admission day for each patient within the allowed range of their release and due dates.
- Assign one nurse to each occupied room on every day of the scheduling period.

The objective is to minimize the total admission delay, defined as the sum of the differences between the admission day and the release date of patients.


**Sumbmission instructions:**

- Each group can be formed by maximum 3 students
- One sumbission per group
- Submission consists of uploading a .ipynb file with the following name format: "LastName1_LastName2_LastName3.ipynb" where the name of the file contains the last names of the students in alphabetic order.
- The submitted file must contain:
  - Name of students
  - Python code using mip library with the model and print out of the objective value in one single code cell
- Questions regarding the project description will be answered in the Forum "Big project" (Please feel free to email me in case the answer is not addressed in the forum at maximiliano.cubillos@polimi.it, after the holiday period)
- Submission deadline:  February 10th 00:00 hrs. No late admissions will be considered.

**Student names**


- Andrea Bellani

In [1]:
# When using Colab, make sure you run this instruction beforehand
!pip install --upgrade cffi==1.15.0
import importlib
import cffi
importlib.reload(cffi)
!pip install mip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.1/484.1 kB 785.9 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for cffi: filename=cffi-1.15.0-cp312-cp312-linux_x86_64.whl size=400944 sha256=7a2e4a029266ec768cf7e12011e27c3459a178c814b8dc3e5e285e1506777c02
  Stored in directory: /root/.cache/pip/wheels/b9/d6/15/0950847bf7d74ea5f0380b8b23a1d81b45bdf48488b4b8237a
Successfully built cffi
  Attempting uninstall: cffi
    Found existing installation: cffi 2.0.0
    Uninstalling cffi-2.0.0:
      Successfully uninstalled cffi-2.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pygit2 1.19.2 requires cffi>=2.0, but you have cffi 1.15.0 which is incompatible.
rpy2 3.5.17 requires cffi>=1.15.1, but you have cffi 1.15.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import mip
import json

def loadDaysInfos (data, days):
    for i in range (data["days"]):
        days.append(i)

    return

def loadPatientsInfos (data, patients, r, d, l):
    for patient in data["patients"]:
        patients.append(patient["id"])
        r.append(patient["release_date"])
        d.append(patient["due_date"])
        l.append (patient["length_of_stay"])

    return

def loadRoomsInfos (data, rooms, z, c):
    for room in data["rooms"]:
        rooms.append(room["id"])
        c.append(room["capacity"])

    for patient in data["patients"]:
        z.append([])
        for room in data["rooms"]:
            if (room["id"] in patient["incompatible_room_ids"]):
                z[len(z)-1].append(0)
            else:
                z[len(z)-1].append(1)

    return

def loadNursesInfos (data, nurses, days, w):
    for nurse in data["nurses"]:
        nurses.append(nurse["id"])
        w.append([])
        for d in days:
            if ({"day" : d} in nurse["working_shifts"]):
                w[len(w)-1].append(1)
            else:
                w[len(w) - 1].append(0)

    return

# constraints additions functions
def rangeForAs(model, a, patients, r, d): # each "a" must be in the range "release date |-| due date" (note that this is not necessary but decreases the optimization complexity a lot
    for i in patients:
        model += a[i] >= r[i]
        model += a[i] <= d[i]
    return

def eachPatientForOneRoomAndLDays(model, x, patients, rooms, days, l): # if a patient stays in a room, it must stay in it for "l" days (and it stays in the hospital for exactly "l" days)
    for i in patients:
        model += mip.xsum(x[i][j][k] for j in rooms for k in days) == l[i]
        for j in rooms:
            for k in days:
                model += l[i]*x[i][j][k] <= mip.xsum(x[i][j][t] for t in days)
    return

def noViolateRoomMaximumCapacity(model, x, patients, rooms, days, c): # room capacities must not be violated
    for k in days:
        for j in rooms:
            model += mip.xsum(x[i][j][k] for i in patients) <= c[j]

    return

def noPatientIncompatibleAssignments(model, x, patients, rooms, days, z): # if a patient is assigned to one room, it must be compatible with it
    for i in patients:
        for j in rooms:
            for k in days:
                model += x[i][j][k] <= z[i][j]

    return

def onlyOneNurseForEachRoomInEachDay(model, y, nurses, rooms, days): # in every day, each room has one and only one nurse assigned
    for k in days:
        for j in rooms:
            model += mip.xsum(y[i][j][k] for i in nurses) == 1

    return

def noNurseUnavailableAssignment(model, y, nurses, rooms, days, w): # if a nurse is assigned to a room in a day, it must be available on that day
    for i in nurses:
        for k in days:
            for j in rooms:
                model += y[i][j][k] <= w[i][k]

    return

def maxThreeRoomsForEachNursePerDay(model, y, nurses, rooms, days): # each nurse must be assigned to a maximum of three rooms each day
    for i in nurses:
        for k in days:
            model += mip.xsum(y[i][j][k] for j in rooms) <= 3

    return

def noPatientUnavailableAssignment(model, x, patients, rooms, days, r, d): # each patient must be in hospital from its release date to its due date
    for i in patients:
        for k in days:
            model += r[i] <= mip.xsum(x[i][j][k] for j in rooms)*k + (1 -mip.xsum(x[i][j][k] for j in rooms))*len(days)
            model += mip.xsum(x[i][j][k] for j in rooms)*k <= d[i]

    return

def onlyConsequentDays(model, x, patients, rooms, days): # any patient stays in hospital in consequent days (mathematical explanation: if in the tensor "x", projected for a patient, there "1 0", any further element must be "0" and if there is "0 1", any previous element must be "0")
    for i in patients:
        for k in range(0, len(days)-1):
            model += mip.xsum(mip.xsum(x[i][j][t] for j in rooms) for t in range(k+1, len(days))) <= (1-mip.xsum(x[i][j][k] for j in rooms))*len(days)+mip.xsum(x[i][j][k+1] for j in rooms)*len(days)
        for k in range(1, len(days)):
            model += mip.xsum(mip.xsum(x[i][j][t] for j in rooms) for t in range(0, k)) <= (1-mip.xsum(x[i][j][k] for j in rooms))*len(days)+mip.xsum(x[i][j][k-1] for j in rooms)*len(days)

    return
def linkingConstraints(model, x, a, patients, rooms, days, l): # ensures that "a" is exactly the first day chosen by "x" (for any patient)
    for i in patients:
        for k in range(0, len(days)-l[i]+1):
            model += (mip.xsum(mip.xsum(x[i][j][t] for j in rooms) for t in range(k,k+l[i])) - l[i] + 1)*k <= a[i]
    return

# json opening
file_path = "test02.json"

with open(file_path, 'r') as f:
    data = json.load(f)

# set definitions
days = []
patients = []
rooms = []
nurses = []

loadDaysInfos(data, days)

# other relevant information definition
r = [] # patients release dates
d = [] # patients due dates
l = [] # patients lengths of stay
loadPatientsInfos(data, patients, r, d, l)

z = [] # patient compatibility for a room (logic)
c = [] # rooms maximum capacities
loadRoomsInfos(data, rooms, z, c)

w = [] # nurse availability for a day (logic)
loadNursesInfos(data, nurses, days, w)

model = mip.Model()

# variables definition
#   x[i][j][k] = 1 <=> patient "i" stays in room "j" during day "t"
#   y[i][j][k] = 1 <=> nurse "i" is assigned to room "j" during day "t"
#   a[i] : entering date of patient "i" (necessary to linearize the bottleneck objective)

x = [[[model.add_var(var_type=mip.BINARY) for k in range(len(days))] for j in range(len(rooms))] for i in range(len(patients))]
y = [[[model.add_var(var_type=mip.BINARY) for k in range(len(days))] for j in range(len(rooms))] for i in range(len(nurses))]
a = [model.add_var(var_type=mip.INTEGER) for i in range(len(patients))]
model.objective = mip.minimize(mip.xsum(a[i]-r[i] for i in range(len(patients))))

# constraints

# variables domain constraints
rangeForAs(model, a, patients, r, d)

# patients constraints
eachPatientForOneRoomAndLDays(model, x, patients, rooms, days, l)
noPatientIncompatibleAssignments(model, x, patients, rooms, days, z) # identical in small project
onlyConsequentDays(model, x, patients, rooms, days)
noPatientUnavailableAssignment(model, x, patients, rooms, days, r, d)

# rooms constraints
noViolateRoomMaximumCapacity(model, x, patients, rooms, days, c) # identical in small project

# nurses constraints
onlyOneNurseForEachRoomInEachDay(model, y, nurses, rooms, days) # identical in small project
noNurseUnavailableAssignment(model, y, nurses, rooms, days, w) # identical in small project
maxThreeRoomsForEachNursePerDay(model, y, nurses, rooms, days)

# linking constraints
linkingConstraints(model, x, a, patients, rooms, days, l)

model.optimize()

print(f"Total admission delay = {model.objective_value}")

Total admission delay = 0.0
